<center><h1>Практическая работа №4</h1></center>

<center><h2>Тема работы: "SQL-аналитика данных интернет-магазина средствами Python"</h2></center>

<h5>Цель работы: научиться применять связку PostgreSQL и Python для аналитики данных. Извлекать данные с помощью SQL, выполнять агрегацию на стороне БД и проводить анализ в Python.</h5>

<h5>Ход работы:</h5>

In [3]:
import psycopg2
import statistics

from dotenv import load_dotenv
import os

load_dotenv()
DB_URI = os.getenv('DB_URI')

try:
    conn = psycopg2.connect(DB_URI)
    cursor = conn.cursor()
    print("Успешное подключение к БД")
except Exception as e:
    print(f"Ошибка подключения: {e}")

Успешное подключение к БД


<h5>1. Рассчет ключевых метрик с использованием SQL.</h5>

In [4]:
query_metrics = """
SELECT
    COUNT(order_id) AS total_orders,
    SUM(price * quantity) AS total_revenue,
    SUM(price * quantity) / COUNT(order_id) AS avg_ticket,
    COUNT(DISTINCT user_id) AS unique_users
FROM orders;
"""

cursor.execute(query_metrics)
metrics = cursor.fetchone()

print("Ключевые метрики")
print("-" * 30)
print(f"Общее количество заказов: {metrics[0]}")
print(f"Суммарная выручка: {metrics[1]:.2f}")
print(f"Средний чек: {metrics[2]:.2f}")
print(f"Уникальных пользователей: {metrics[3]}")

Ключевые метрики
------------------------------
Общее количество заказов: 500
Суммарная выручка: 372097.35
Средний чек: 744.19
Уникальных пользователей: 30


<h5>2. Сегментный анализ.</h5>

In [5]:
query_segments = """
SELECT
    category,
    COUNT(order_id) AS order_count,
    SUM(price * quantity) AS revenue
FROM orders
GROUP BY category
ORDER BY revenue DESC;
"""

cursor.execute(query_segments)
categories = cursor.fetchall()

print("-" * 40)
print(f"{'Категория':<15} | {'Заказы':<8} | {'Выручка':<10}")
print("-" * 40)

for row in categories:
    category, count, rev = row
    print(f"{category:<15} | {count:<8} | {rev:<10.2f}")
print("-" * 40)

----------------------------------------
Категория       | Заказы   | Выручка   
----------------------------------------
Books           | 111      | 88284.07  
Electronics     | 97       | 75467.72  
Home & Garden   | 93       | 72211.18  
Clothing        | 96       | 68089.61  
Sports          | 103      | 68044.77  
----------------------------------------


<h5>3. Анализ пользователей.</h5>

In [6]:
cursor.execute("SELECT SUM(price * quantity), COUNT(order_id) FROM orders")
total_data = cursor.fetchone()
total_revenue = float(total_data[0])
total_orders_count = int(total_data[1])

query_users = """
SELECT
    user_id,
    COUNT(order_id) AS orders_made,
    SUM(price * quantity) AS total_spent
FROM orders
GROUP BY user_id
ORDER BY total_spent DESC;
"""

cursor.execute(query_users)
top_users = cursor.fetchall()

print("-" * 60)
print(f"{'Клиент':<10} | {'Заказы':<8} | {'Траты':<12} | {'Доля от выручки (%)':<20}")
print("-" * 60)

for user_id, orders_made, total_spent in top_users:
    share = (float(total_spent) / total_revenue) * 100
    print(f"{user_id:<10} | {orders_made:<8} | {total_spent:>12.2f} | {share:>18.2f}%")

print("-" * 60)
print(f"{'ИТОГО:':<10} | {total_orders_count:<8} | {total_revenue:>12.2f} | {'100.00%':>18}")
print("-" * 60)

------------------------------------------------------------
Клиент     | Заказы   | Траты        | Доля от выручки (%) 
------------------------------------------------------------
USER-13    | 19       |     19573.45 |               5.26%
USER-3     | 19       |     17537.87 |               4.71%
USER-27    | 18       |     17386.55 |               4.67%
USER-8     | 19       |     17309.17 |               4.65%
USER-17    | 21       |     16785.96 |               4.51%
USER-2     | 19       |     16691.57 |               4.49%
USER-23    | 20       |     16378.40 |               4.40%
USER-16    | 22       |     16043.71 |               4.31%
USER-14    | 14       |     15156.01 |               4.07%
USER-25    | 19       |     14296.43 |               3.84%
USER-21    | 21       |     14282.10 |               3.84%
USER-1     | 22       |     13512.74 |               3.63%
USER-10    | 24       |     13399.62 |               3.60%
USER-24    | 16       |     13071.69 |             

<h5>4. Временной анализ.</h5>

In [7]:
query_temporal = """
SELECT
    order_date,
    SUM(price * quantity) AS daily_revenue
FROM orders
GROUP BY order_date
ORDER BY order_date ASC;
"""

cursor.execute(query_temporal)
daily_data = cursor.fetchall()

if daily_data:
    sorted_by_revenue = sorted(daily_data, key=lambda x: x[1])

    bottom_5 = sorted_by_revenue[:5]

    top_5 = sorted_by_revenue[-5:][::-1]

    print("-" * 38)
    print(f"{'Дата':<12} | {'Выручка за день (₽)':<20}")
    print("-" * 38)

    print("Периоды с максимальными значениями:")
    for date, revenue in top_5:
        print(f"{str(date):<12} | {float(revenue):>20.2f}")

    print("-" * 38)
    print("Периоды с минимальными значениями:")
    for date, revenue in bottom_5:
        print(f"{str(date):<12} | {float(revenue):>20.2f}")
    print("-" * 38)

    max_rev = float(top_5[0][1])
    min_rev = float(bottom_5[0][1])

    print(f"\nРазрыв между пиковым и минимальным значением составляет {max_rev - min_rev:.2f} руб.")

--------------------------------------
Дата         | Выручка за день (₽) 
--------------------------------------
Периоды с максимальными значениями:
2024-03-27   |             11707.34
2024-01-30   |             10122.52
2024-01-29   |              9487.26
2024-03-21   |              7837.83
2024-03-17   |              7815.46
--------------------------------------
Периоды с минимальными значениями:
2024-02-27   |               444.92
2024-02-11   |               528.82
2024-01-26   |               937.73
2024-03-09   |               977.45
2024-03-10   |               984.49
--------------------------------------

Разрыв между пиковым и минимальным значением составляет 11262.42 руб.


<h5>5. Рассчет статистических показателей.</h5>

In [8]:
revenues = [float(row[1]) for row in daily_data]

if revenues:
    mean_rev = statistics.mean(revenues)
    stdev_rev = statistics.stdev(revenues) if len(revenues) > 1 else 0

    # Все, что выше или ниже этих границ, считается нетипичным поведением
    lower_bound = mean_rev - 2 * stdev_rev
    upper_bound = mean_rev + 2 * stdev_rev

    print("-" * 52)
    print("Статистические показатели выручки")
    print("-" * 52)
    print(f"Средняя дневная выручка:  {mean_rev:>12.2f} ₽")
    print(f"Стандартное отклонение:   {stdev_rev:>12.2f} ₽")
    print(f"Доверительный интервал:   {max(0, lower_bound):.2f} — {upper_bound:.2f} ₽")
    print("-" * 52)

    anomalies = []
    for date, rev in daily_data:
        rev_float = float(rev)
        if rev_float > upper_bound or rev_float < lower_bound:
            anomalies.append((date, rev_float))

    print("\nВыявленные аномалий в продажах:")
    if anomalies:
        print(f"{'Дата':<12} | {'Выручка (₽)':<15} | {'Тип отклонения'}")
        print("-" * 52)
        for d, r in anomalies:
            status = "Всплеск (High)" if r > mean_rev else "Просадка (Low)"
            print(f"{str(d):<12} | {r:>12.2f}    | {status}")
        print("-" * 52)
    else:
        print("В анализируемом периоде значимых отклонений не обнаружено.")

cursor.close()
conn.close()
print("\nСоединение с базой данных закрыто.")

----------------------------------------------------
Статистические показатели выручки
----------------------------------------------------
Средняя дневная выручка:       4134.41 ₽
Стандартное отклонение:        2134.63 ₽
Доверительный интервал:   0.00 — 8403.68 ₽
----------------------------------------------------

Выявленные аномалий в продажах:
Дата         | Выручка (₽)     | Тип отклонения
----------------------------------------------------
2024-01-29   |      9487.26    | Всплеск (High)
2024-01-30   |     10122.52    | Всплеск (High)
2024-03-27   |     11707.34    | Всплеск (High)
----------------------------------------------------

Соединение с базой данных закрыто.


<h5>Вывод: в ходе выполнения работы была успешно настроена и применена связка СУБД PostgreSQL и Python. Использование агрегатных функций SQL (SUM, COUNT, GROUP BY) позволило перенести тяжелые вычислительные операции на сторону базы данных, сэкономив ресурсы приложения..</h5>